# [LAB] Langfuse로 실행 기록하고 평가하기

LLM 애플리케이션의 실행을 Langfuse에 기록하고(Trace), 답변 품질에 점수를 남겨(Score) 봅니다. 셀의 출력은 기록을 붙이든 안 붙이든 똑같이 보입니다. 그래서 이 노트북의 결과는 대부분 **Langfuse 화면**에서 확인합니다. 셀을 실행한 뒤 브라우저의 Langfuse 화면을 함께 봅니다.

- **2장 체인 실행 기록하기**: 체인을 실행할 때 기록 도구(CallbackHandler)를 넘깁니다.
- **3장 그래프 실행 기록하기**: LangGraph 그래프에 기록 도구를 기본 설정으로 붙입니다.
- **4장 체인 밖 함수 기록하기**: `@observe`로 검색 같은 일반 함수도 같은 기록에 넣습니다.
- **5~7장 점수 남기기**: 규칙, LLM 채점(LLM-as-a-Judge), 사용자 피드백으로 점수를 남깁니다.
- **8장 RAG 답변을 세 기준으로 채점하기**: 정확성·관련성·완성도를 LLM이 채점합니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다.

**Langfuse 화면 열기** 브라우저에서 `.env`의 `LANGFUSE_BASE_URL` 주소를 열어 로그인하고, 키를 받은 프로젝트를 엽니다.

**내 trace 찾기** Langfuse 화면 왼쪽 메뉴의 **Tracing**을 엽니다. 목록은 가장 최근 것이 맨 위에 있습니다. 여러 사람이 같은 프로젝트를 쓰면 남의 기록도 섞여 보이므로, 다음 세 가지로 내 것을 고릅니다.

1. **이름**: 목록의 이름 칸에 보이는 이름으로 검색 칸에서 거릅니다. 이 노트북의 코드는 장마다 이름이 정해져 있습니다(각 장에서 알려 드립니다).
2. **시각**: 셀을 실행한 시각과 같은 시각의 기록을 봅니다.
3. **입력**: 기록을 열어 입력(Input)에 내가 보낸 질문이 있는지 봅니다.

기록은 몇 초 늦게 도착합니다. 셀 끝의 `flush()`까지 실행한 뒤 화면을 새로고침합니다. 코드 출력의 `대시보드`는 Langfuse 화면을, `[Scores] 탭`은 왼쪽 메뉴의 **Scores**를 말합니다.

In [ ]:
%cd ../../../course/Chapter02/notebooks

**목차**

- 1. 실습 방법
- 2. 체인 실행 기록하기
  - 2.1 기록 도구와 체인 준비
  - 2.2 질문 하나 기록하기
  - 2.3 질문 여러 개 기록하기
- 3. 그래프 실행 기록하기
  - 3.1 날씨 도구와 Agent 노드
  - 3.2 그래프 조립과 기록 도구 붙이기
  - 3.3 실행하기
- 4. 체인 밖 함수 기록하기
  - 4.1 검색·정리 함수에 이름 붙이기
  - 4.2 RAG 흐름 묶기
  - 4.3 실행하기
- 5. 규칙으로 채점하기
  - 5.1 답변 체인
  - 5.2 예시 포함 여부 채점하기
- 6. LLM이 채점하기
  - 6.1 LLM Judge로 채점하기
- 7. 사용자 피드백 기록하기
  - 7.1 사용자 피드백 점수 남기기
- 8. RAG 답변을 세 기준으로 채점하기
  - 8.1 문서 세 개
  - 8.2 채점 결과 형식
  - 8.3 검색과 답변 생성 기록하기
  - 8.4 세 기준으로 채점하는 Judge
  - 8.5 테스트 케이스
  - 8.6 실행하기
- 9. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env(["LANGFUSE"])

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

**Langfuse에 연결하기 전에: 회사 인증서 쓰기**

Langfuse(`cloud.langfuse.com`)는 인터넷에 있는 서버라, 기록을 보낼 때 HTTPS로 연결합니다. 사내망에서는 이 연결이 회사 보안 인증서를 거칩니다. 그런데 파이썬은 기본으로 자기가 가진 인증서 목록만 믿기 때문에, 회사 인증서를 모르고 연결을 거부할 수 있습니다. 이때 오류에 `SSLError`나 `CERTIFICATE_VERIFY_FAILED`가 보입니다.

`truststore`는 파이썬이 **PC(운영체제)에 등록된 인증서**를 믿게 해 줍니다. 회사 PC에는 회사 인증서가 등록되어 있으니 연결이 됩니다. 회사 밖 PC에서 실행해도 문제없습니다.

- 아래 셀은 **Langfuse를 쓰는 셀보다 먼저** 한 번 실행합니다.
- 커널을 다시 시작했으면 이 셀부터 다시 실행합니다.

In [ ]:
import truststore

# 파이썬이 PC에 등록된 인증서(회사 인증서 포함)를 믿게 합니다
truststore.inject_into_ssl()
print("🟢 PC 인증서 사용 설정 완료")

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

도구를 부를 때 `Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')` 같은 줄이 섞여 보일 수 있습니다. 디버그 내용을 출력하는 쪽의 문제라 실행 결과에는 영향이 없으니 그대로 두면 됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

<details><summary>강사용 메모</summary>

- 원본: `course/Chapter02/code/raw/05-observability/` 다섯 파일. 01~03·05는 raw(solution과 docstring 경로 한 줄만 다름), 04는 solution을 씁니다(4-2 목차 3절). 파일 맨 위 docstring(클라우드 주소, 실행 경로)은 노트북에서 뺐습니다.
- **Langfuse 서버**: 받은 코드와 `.env.sample`은 클라우드(`jp.cloud.langfuse.com`), 강사 맥에는 self-host(`infra/langfuse/`)가 있습니다. 강의장에서 어느 주소를 쓸지, 수강생마다 프로젝트(키)를 따로 줄지 한 프로젝트를 같이 쓸지 **미정**입니다. 같이 쓰면 위 "내 trace 찾기"(이름·시각·입력)로 고릅니다(2026-09-28 사용자 결정 3).
- 키 확인: 2장 준비 코드에서 `get_client().auth_check()`를 부릅니다(사용자 결정 3). 키가 틀리면 Langfuse는 경고만 내고 코드는 정상 실행되어 "조용히 틀린 데모"가 됩니다. 환경 점검 셀(`check_env(["LANGFUSE"])`)은 키 **이름**만 봅니다.
- Langfuse 화면 메뉴 이름(Tracing, Scores)과 검색 칸 동작은 받은 코드의 출력 문구와 일반 화면 기준으로 적었습니다. 강의장에서 쓸 Langfuse 버전 화면으로는 **미확인**입니다. 클라우드 새 조직은 공개 API `GET /api/public/traces`가 막혀 있었습니다(2026-09-28, 410 LEGACY_API). 화면 확인은 영향 없음.
- 사외 실측(2026-09-28, `COURSE_ENV=out`, OpenAI, jp 클라우드): 01 약 16초, 02 약 8초, 03 약 8초, 04 약 13초, 05 약 10초. 사내 게이트웨이 시간은 미확인입니다.
- 사외 실측 trace 이름: 01·04 `RunnableSequence`, 02 `LangGraph`, 03·05 `rag-pipeline`.
- 11차수 피드백: 뒷자리에서 Langfuse 화면이 안 보였습니다. 성공 기준은 수강생 각자 PC 화면에서 확인하도록 썼고, 강사 화면은 확대합니다.
- 2026-10-21 사용자 결정으로 객관식(보기 고르기)을 없애고 M1과 같은 값 바꾸기판으로 바꿨습니다(코드는 완성, CHANGE_HERE 다섯 곳: 4.1 이름, 5.1 예시 지시, 6.1 Judge 기준, 7.1 피드백 값, 8.5 CASE C 문서). "무슨 일이 생기나"는 사외 OpenRouter `qwen/qwen3-235b-a22b-2507`(OpenAI 크레딧 소진으로 대체)와 cloud Langfuse 실행 결과입니다. 노트북 전체 새 커널 실행 오류 0. 이 모델로는 CASE B(문서 없음)도 아는 내용으로 답해 세 점수 1.0이 나왔습니다(1회).
- 2026-10-21 사용자 OK: 5.1 '예시 지시 지우기'는 점수가 그대로라 뺐다. 6.1은 차이가 작던 문장 대신 '채점 기준: 답변이 50자를 넘으면 무조건 0.0을 출력합니다.'로 바꿈(OpenRouter Qwen, 0.95 → 0.0, 3/3). 약한 기준('초등학생도…', '한 문장을 넘으면 0점')은 이 모델이 무시했다(0.95·1.0).

</details>

## 1. 실습 방법

코드는 모두 완성되어 있습니다. 위에서부터 차례로 실행합니다.

- `# ✍️ CHANGE_HERE:` 주석이 있는 셀은 먼저 그대로 실행해 결과를 봅니다. 그다음 셀 위의 **바꿔 볼 값**을 복사해, 주석 바로 아래 줄의 표시된 자리(지금 값)를 지우고 붙여 넣습니다. 어떻게 달라질지 생각해 본 뒤 다시 실행합니다.
- 달라진 결과를 본 뒤 셀 아래 접힌 **무슨 일이 생기나**를 펼쳐 비교합니다. 확인했으면 값을 원래대로 되돌립니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. 체인 실행 기록하기

프롬프트 → 모델 → 출력 파서로 이어진 체인을 실행하면서 Langfuse에 기록을 남깁니다. 기록 도구는 `CallbackHandler`입니다. `.env`의 Langfuse 설정을 알아서 읽으므로 만들기만 하면 되고, 체인을 실행할 때 `config`의 `callbacks` 목록에 넣어 넘기면 그 실행의 모든 단계가 기록됩니다.

`invoke`를 한 번 호출할 때마다 trace가 하나씩 생깁니다. 이 장은 질문 하나, 그다음 질문 두 개를 실행하므로 trace가 세 개 생깁니다. 이 장의 trace 이름은 `RunnableSequence`입니다(체인의 기본 이름).

답이 나왔다고 기록이 남은 것은 아닙니다. 셀을 실행한 뒤 Langfuse 화면에서 trace 수를 셉니다.

**왜 배우나:** 실행 결과만 출력해서는 모델에 실제로 어떤 프롬프트가 갔는지, 어느 단계가 오래 걸렸는지 알 수 없습니다. 기록 도구를 붙이는 것이 관찰의 첫걸음입니다.

준비 코드입니다. 필요한 도구를 불러오고, **Langfuse에 연결되는지 확인**합니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다. 출력에 🟢가 나오면 다음으로 넘어갑니다. ⛔가 나오면 `.env`의 Langfuse 값 세 개를 확인하고, 이 셀의 출력을 AI에게 보여 원인을 물어봅니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client
from langfuse.langchain import CallbackHandler

from config.llm import get_llm

load_dotenv()

# Langfuse 서버에 연결되고 키가 맞는지 확인합니다.
# 키가 틀려도 아래 실습 코드는 오류 없이 돌아가므로, 여기서 먼저 봅니다.
try:
    langfuse_ok = get_client().auth_check()
except Exception as e:
    langfuse_ok = False
    print(f"오류 종류: {type(e).__name__}")
if langfuse_ok:
    print("🟢 Langfuse 연결 확인: 키가 맞습니다.")
else:
    print("⛔ Langfuse에 연결하지 못했습니다. .env의 LANGFUSE_ 값 세 개를 확인하세요.")

<details><summary>강사용 메모</summary>

- 원본 01은 TODO 두 곳이 모두 같은 문장("config의 callback으로 `langfuse_handler`를 전달")입니다. 슬라이드 CallbackHandler 쪽 코드와 같은 모양이라 과제로 냈습니다.

</details>

### 2.1 기록 도구와 체인 준비

In [ ]:
# CallbackHandler: .env의 LANGFUSE_* 환경변수를 자동으로 읽어 인증
# 이 핸들러를 chain.invoke()에 전달하면 모든 단계가 Langfuse에 자동 기록됨
langfuse_handler = CallbackHandler()

llm = get_llm()
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "한국어로 간결하게 답변하세요."),
        ("human", "{question}"),
    ]
)
chain = prompt | llm | StrOutputParser()

### 2.2 질문 하나 기록하기

질문 하나를 체인에 넣어 실행하고, 답변을 `result`에 담습니다. 이때 위에서 만든 `langfuse_handler`를 함께 넘겨야 이 실행이 Langfuse에 기록됩니다. 질문은 `LangGraph와 LangChain의 차이를 한 줄로 설명해줘`로 하고, 위에서 만든 프롬프트의 `{question}` 자리에 넣습니다.

In [ ]:
print("=== 단일 질문 트레이싱 ===")
# config={"callbacks": [...]} — LangChain이 실행 중 각 단계마다 핸들러를 호출함
result = chain.invoke(
    {"question": "LangGraph와 LangChain의 차이를 한 줄로 설명해줘"},
    config={"callbacks": [langfuse_handler]},
)
print(result)
print()

### 2.3 질문 여러 개 기록하기

질문 두 개를 차례로 실행합니다. 반복문 안에서 질문(`q`)마다 체인을 실행해 `answer`에 담고, 이번에도 `langfuse_handler`를 넘깁니다. 셀 끝의 `flush()`는 모아 둔 기록을 Langfuse 서버로 바로 보냅니다.

In [ ]:
print("=== 여러 질문 배치 트레이싱 ===")
questions = [
    {"question": "RAG가 뭔가요?"},
    {"question": "Vector DB는 왜 쓰나요?"},
]
# 각 invoke마다 별도의 trace로 기록됨
for q in questions:
    answer = chain.invoke(q, config={"callbacks": [langfuse_handler]})
    print(f"Q: {q['question']}")
    print(f"A: {answer}")
    print()

# flush: 비동기로 큐에 쌓인 이벤트를 Langfuse 서버로 전송 완료 후 종료
# 스크립트 종료 직전에 호출하지 않으면 일부 데이터가 누락될 수 있음
langfuse = get_client()
langfuse.flush()
print("Tracing에서 확인")

**볼 지점**
- 셀 출력은 기록을 붙이기 전과 똑같습니다. 기록이 남았는지는 Langfuse 화면에서만 알 수 있습니다.
- trace를 열면 체인의 세 단계가 트리로 보입니다. 모델 호출 단계에는 실제로 모델에 간 메시지(시스템 메시지 "한국어로 간결하게 답변하세요."와 질문)가 들어 있습니다.
- 모델 호출 단계에서 걸린 시간과 토큰 수도 확인합니다. 비용이 어디서 생기는지 보는 자리입니다.

<details><summary>강사용 메모</summary>

- 모델 호출 단계 이름은 사외(OpenAI)에서 `ChatOpenAI`였습니다. 사내 게이트웨이 모델 클래스 이름은 미확인입니다(성공 기준의 `ChatOpenAI` 괄호는 "모델 호출"로 읽어 주세요).

</details>

## 3. 그래프 실행 기록하기

날씨 도구를 쓰는 LangGraph Agent를 기록합니다. 그래프는 agent 노드(모델)와 tools 노드(도구 실행)를 오가며 답을 만듭니다.

앞 장에서는 `invoke`를 호출할 때마다 핸들러를 넘겼습니다. 이번에는 그래프를 만들 때 `with_config`로 핸들러를 **기본 설정**으로 붙입니다. 그러면 이후 `invoke`에는 아무것도 넘기지 않아도 모든 실행이 기록됩니다. 이 장의 trace 이름은 `LangGraph`입니다.

그래프를 만드는 셀은 아무것도 출력하지 않으니, 아래 "실행하기" 셀까지 실행하고 Langfuse 화면을 봅니다.

**왜 배우나:** Agent는 모델과 도구를 여러 번 오갑니다. 어느 노드에서 어떤 도구를 어떤 입력으로 불렀는지, 어디서 시간이 걸렸는지를 노드 순서대로 볼 수 있어야 틀린 답의 원인을 가를 수 있습니다.

준비 코드입니다. 그래프와 도구에 필요한 것을 불러옵니다. 실행만 합니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv();

<details><summary>강사용 메모</summary>

- 원본 02 raw 사용. `bind_tools`를 씁니다. 사내 게이트웨이의 도구 호출 동작은 M1 2-1과 같은 조건이며 이 권에서는 미확인입니다.
- 사외 실측: "부산이랑 제주" 질문에서 `get_weather`가 두 번(부산, 제주) 호출되었습니다.

</details>

### 3.1 날씨 도구와 Agent 노드

In [ ]:
@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 반환한다."""
    data = {"서울": "맑음 23°C", "부산": "흐림 19°C", "제주": "비 17°C"}
    return data.get(city, f"{city}의 날씨 정보 없음")


tools = [get_weather]
llm = get_llm().bind_tools(tools)


class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

### 3.2 그래프 조립과 기록 도구 붙이기

노드와 엣지를 연결한 `builder`를 실행할 수 있는 그래프로 만들고(`compile`), 거기에 `with_config`로 `langfuse_handler`를 기본 설정으로 붙여 `app`에 담습니다. 이 셀은 아무것도 출력하지 않습니다. 결과는 아래 "실행하기" 셀에서 봅니다.

In [ ]:
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

# with_config(): 그래프 인스턴스에 기본 config를 고정
# 이후 모든 invoke() / stream() 호출에 CallbackHandler가 자동 적용됨
langfuse_handler = CallbackHandler()
app = builder.compile().with_config({"callbacks": [langfuse_handler]})

### 3.3 실행하기

In [ ]:
print("=== LangGraph 트레이싱 ===")
inputs = [
    "서울 날씨 알려줘",
    "부산이랑 제주 날씨 비교해줘",
]
for question in inputs:
    tracing_graph_result = app.invoke({"messages": [HumanMessage(content=question)]})
    results = tracing_graph_result["messages"]
    for m in results:
        m.pretty_print()
    print()

langfuse = get_client()
langfuse.flush()
print("Tracing에서 확인")

**볼 지점**
- `LangGraph` trace 안에 `agent`, `tools_condition`, `tools`, `get_weather`가 실행 순서대로 보입니다. `tools_condition`은 도구를 부를지 끝낼지 정한 갈림길입니다.
- `get_weather` 단계를 누르면 모델이 도구에 넘긴 입력(도시 이름)과 도구가 돌려준 날씨가 보입니다. "부산이랑 제주" 질문에서는 도구가 몇 번 호출되었는지 셉니다.
- 두 번째 `agent` 단계의 모델 호출이 도구 결과를 받아 최종 답을 씁니다.
- 셀 출력에서도 같은 흐름이 보입니다. `Ai Message` 칸의 `Tool Calls:`가 도구 호출이고, `Tool Message` 칸이 `get_weather`가 돌려준 날씨입니다.

<details><summary>강사용 메모</summary>

- 원본은 마지막 답만 `print(...content)`로 출력합니다. 노트북은 결과 변수를 그래프가 드러나는 이름(`tracing_graph_result`)으로 바꾸고, 메시지 목록을 `results`로 꺼내 `pretty_print()`로 출력합니다(2026-10-17 규칙, dev/AGENTS.md).

</details>

## 4. 체인 밖 함수 기록하기

RAG처럼 "문서 검색 → 문맥 정리 → 모델 호출"로 이어지는 흐름을 기록합니다. 검색과 문맥 정리는 LangChain 체인이 아닌 일반 Python 함수라서 CallbackHandler로는 보이지 않습니다. 이런 함수에는 `@observe`를 붙입니다. `name`에 적은 이름이 Langfuse 화면에 표시됩니다.

`@observe`를 붙인 함수 안에서 다른 `@observe` 함수나 체인을 호출하면, 그 실행이 부모 아래 자식으로 묶여 트리가 됩니다. 이 장의 trace 이름은 가장 바깥 함수의 이름 `rag-pipeline`입니다. 검색은 실제로 하지 않고 정해 둔 문서 두 개를 돌려줍니다.

코드는 완성되어 있습니다. 먼저 끝까지 실행하고, `# ✍️ CHANGE_HERE:` 줄을 바꿔 다시 실행합니다.

준비 코드입니다. `observe`를 불러오고 Langfuse 클라이언트, 기록 도구, 모델을 준비합니다. 실행만 합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client, observe
from langfuse.langchain import CallbackHandler

from config.llm import get_llm

load_dotenv()

langfuse = get_client()
langfuse_handler = CallbackHandler()
llm = get_llm()

### 4.1 검색·정리 함수에 이름 붙이기

**✍️ CHANGE_HERE: 검색 함수의 이름(name)을 바꾸면 Langfuse의 트리가 어떻게 보일까요?**

**바꿔 볼 값** — 바로 아래 줄의 `retrieve-documents` 자리에 복사해 붙여 넣기

hong 자리에 내 이름(영문)을 넣어도 됩니다
```text
retrieve-hong
```

이 셀부터 4.3 실행하기 셀까지 차례로 다시 실행하고, Langfuse에서 새로 생긴 `rag-pipeline` trace를 엽니다. 확인했으면 `retrieve-documents`로 되돌립니다.

In [ ]:
# @observe(): 이 함수의 호출이 Langfuse에 span으로 기록됨
# name 파라미터로 대시보드에 표시될 이름을 지정할 수 있음
# ✍️ CHANGE_HERE: 검색 함수의 이름(name)을 바꾸면 Langfuse의 트리가 어떻게 보일까요?
@observe(name="retrieve-documents")
def retrieve(query: str) -> list[str]:
    # 실제로는 벡터스토어에서 검색하는 부분
    # 여기서는 시뮬레이션용 더미 데이터 반환
    return [
        f"[문서1] {query}에 관한 첫 번째 참고 자료입니다.",
        f"[문서2] {query}와 관련된 두 번째 참고 자료입니다.",
    ]


@observe(name="format-context")
def format_context(docs: list[str]) -> str:
    return "\n".join(docs)

<details><summary>무슨 일이 생기나</summary>

셀 출력(Q·A)은 바꾸기 전과 같습니다. Langfuse에서 새로 생긴 `rag-pipeline` trace를 열면 검색 단계의 이름이 `retrieve-documents` 대신 `retrieve-hong`으로 보입니다. 조회 API로 그 trace의 단계 이름을 확인하니 `retrieve-hong`, `format-context`, 체인 단계(`ChatPromptTemplate` 등)였습니다.

`name`은 화면에 보일 이름일 뿐 함수가 하는 일은 그대로입니다. 여러 사람이 같은 프로젝트를 쓸 때 내 이름을 넣어 두면 내 기록을 찾기 쉽습니다.

</details>

### 4.2 RAG 흐름 묶기

In [ ]:
# @observe로 감싼 함수 안에서 LangChain 체인을 호출하면
# 체인 실행이 이 span의 자식으로 자동 연결됨
@observe(name="rag-pipeline")
def rag_pipeline(question: str) -> str:
    docs = retrieve(question)
    context = format_context(docs)

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "아래 컨텍스트를 바탕으로 질문에 답하세요.\n\n{context}"),
            ("human", "{question}"),
        ]
    )
    chain = prompt | llm | StrOutputParser()

    # CallbackHandler를 여기서도 전달해야 LLM 호출이 같은 트레이스에 묶임
    return chain.invoke(
        {"context": context, "question": question},
        config={"callbacks": [langfuse_handler]},
    )

### 4.3 실행하기

In [ ]:
print("=== @observe 데코레이터 트레이싱 ===")
questions = ["LangGraph란 무엇인가요?", "RAG의 장점은?"]
for q in questions:
    answer = rag_pipeline(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

langfuse.flush()
print("✓ 대시보드에서 retrieve → format → LLM 호출 트리를 확인하세요.")
print("Tracing에서 확인")

**볼 지점**
- `retrieve-documents`를 누르면 함수의 입력(질문)과 출력(문서 두 개)이 보입니다. `@observe`가 함수의 입력과 반환값을 기록합니다.
- 체인(`RunnableSequence`)은 `rag-pipeline` 아래에 붙어 있습니다. `rag_pipeline` 안에서 체인에 핸들러를 넘겼기 때문입니다.
- 검색·정리·모델 호출이 한 trace 안에 모여 있어서, 답이 이상할 때 검색 결과부터 차례로 따라가 볼 수 있습니다.

## 5. 규칙으로 채점하기

여기부터는 기록(Trace)에 **점수(Score)**를 붙입니다. 점수는 `langfuse.create_score`로 남기고, 어느 trace에 붙일지는 trace ID로 정합니다. 핸들러의 `last_trace_id`에 방금 실행한 trace의 ID가 들어 있습니다.

이 장은 규칙 기반 평가입니다. 답변에 예시가 들어 있는지를 코드로 판단해 1.0 또는 0.0을 `has-example`이라는 이름으로 남깁니다. 답변 체인 `run_chain`은 시스템 메시지로 "반드시 예시를 포함하세요"라고 지시합니다. 5~7장의 trace 이름은 2장과 같은 `RunnableSequence`입니다. 기록을 열어 입력(Input)의 질문으로 내 것을 고릅니다.

코드는 완성되어 있고 이 장은 바꿀 값이 없습니다. 실행하고 점수와 키워드 목록만 확인합니다.

**왜 배우나:** 기록만으로는 답이 좋은지 알 수 없습니다. 점수를 남겨 두면 프롬프트를 바꿨을 때 품질이 나아졌는지 숫자로 비교할 수 있습니다.

준비 코드입니다. Langfuse 클라이언트(`langfuse`)와 모델을 준비합니다. 실행만 합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client
from langfuse.langchain import CallbackHandler

from config.llm import get_llm

load_dotenv()

langfuse = get_client()
llm = get_llm()

<details><summary>강사용 메모</summary>

- 원본: `course/Chapter02/code/solution/05-observability/04_evaluation.py`. raw 102~104행의 이전 강사 메모 주석(병렬 실행·승인 대기, 코드와 무관, 들여쓰기 어긋남)이 없는 solution을 썼습니다.
- 원본은 한 파일에 규칙 기반 → LLM-as-a-Judge → 수동 평가 세 구획이 있습니다. 노트북에서는 구획마다 장을 따로 두고(커리큘럼의 평가 방법 3가지에 대응), 장마다 접힌 준비 코드가 `langfuse`·`llm`·`run_chain`을 다시 만듭니다. 장 끝마다 `langfuse.flush()`를 넣었습니다(원본은 파일 끝에 한 번).
- 원본 주석 `# get_trace_id(): …`는 코드(`handler.last_trace_id`)와 달라 노트북에서만 `# last_trace_id: …`로 고쳤습니다.

</details>

### 5.1 답변 체인

In [ ]:
def run_chain(question: str, handler: CallbackHandler) -> str:
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "한국어로 간결하게 답변하세요. 반드시 예시를 포함하세요."),
            ("human", "{question}"),
        ]
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"question": question}, config={"callbacks": [handler]})

### 5.2 예시 포함 여부 채점하기

In [ ]:
# ── 1. 규칙 기반 평가 ──────────────────────────────────────────────────────────
print("=== 규칙 기반 평가 ===")
handler1 = CallbackHandler()
question1 = "Python 리스트와 튜플의 차이를 설명해줘"
answer1 = run_chain(question1, handler1)
print(f"답변: {answer1}\n")

# last_trace_id: 직전 invoke에서 생성된 trace의 ID
trace_id1 = handler1.last_trace_id

# 예시 포함 여부를 규칙으로 판단해 점수 부여
has_example = any(
    keyword in answer1 for keyword in ["예:", "예를 들어", "예시", "[", "("]
)
langfuse.create_score(
    trace_id=trace_id1,
    name="has-example",  # 대시보드에 표시될 평가 항목 이름
    value=1.0 if has_example else 0.0,
    comment="예시 포함 여부 자동 검사",
)
print(
    f"예시 포함 여부: {'✓' if has_example else '✗'} (score={1.0 if has_example else 0.0})"
)
print()


langfuse.flush()

**볼 지점**
- 점수는 코드가 정합니다. 같은 답변이면 몇 번을 채점해도 같은 점수가 나옵니다(재현성).
- 규칙은 키워드 목록 중 하나라도 답변에 들어 있으면 예시가 있다고 봅니다. 키워드 목록을 읽어 보고, 예시가 아닌데도 걸릴 만한 것이 있는지 찾아봅니다.

<details><summary>강사용 메모</summary>

- 사외 실측: 원본 그대로 `✓ (score=1.0)`. 규칙의 `'('`·`'['`는 괄호만 있어도 참이라, 예시 지시를 빼도 1.0이 나오기 쉽습니다. 수강생이 물으면 이 점을 짚어 줍니다.

</details>

## 6. LLM이 채점하기

이번에는 다른 LLM이 답변 품질을 0.0~1.0 사이 숫자로 채점합니다(LLM-as-a-Judge). 채점 결과를 `llm-judge-quality`라는 이름으로 답변 trace에 남깁니다. Judge 호출에는 핸들러를 넘기지 않아서 Judge 자체는 따로 기록되지 않습니다. 점수만 답변 trace에 붙습니다.

이번에도 먼저 원본 그대로 결과를 본 뒤 `# ✍️ CHANGE_HERE:` 줄을 바꿉니다.

준비 코드입니다. 앞 장과 같은 모델과 답변 체인 `run_chain`을 다시 준비합니다. 실행만 합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client
from langfuse.langchain import CallbackHandler

from config.llm import get_llm

load_dotenv()

langfuse = get_client()
llm = get_llm()


def run_chain(question: str, handler: CallbackHandler) -> str:
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "한국어로 간결하게 답변하세요. 반드시 예시를 포함하세요."),
            ("human", "{question}"),
        ]
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"question": question}, config={"callbacks": [handler]})

### 6.1 LLM Judge로 채점하기

**✍️ CHANGE_HERE: Judge 지시 맨 앞에 채점 기준 한 문장을 더하면 같은 답의 점수가 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `아래 답변의 품질을` 자리에 복사해 붙여 넣기

채점 기준 한 문장을 앞에 더한 값
```text
채점 기준: 답변이 50자를 넘으면 무조건 0.0을 출력합니다. 아래 답변의 품질을
```

이 셀을 다시 실행하고 `LLM 채점 점수:`를 바꾸기 전과 비교합니다. 확인했으면 원래대로 되돌립니다.

In [ ]:
# ── 2. LLM-as-a-Judge 평가 ────────────────────────────────────────────────────
# 별도 LLM이 응답 품질을 0~1로 채점 — 사람 평가 없이 대규모 자동 평가에 사용
print("=== LLM-as-a-Judge 평가 ===")
handler2 = CallbackHandler()
question2 = "딥러닝이란 무엇인가요?"
answer2 = run_chain(question2, handler2)
print(f"답변: {answer2}\n")

trace_id2 = handler2.last_trace_id

judge_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            # ✍️ CHANGE_HERE: Judge 지시 맨 앞에 채점 기준 한 문장을 더하면 같은 답의 점수가 어떻게 달라질까요?
            "아래 답변의 품질을 0.0~1.0 사이 숫자 하나만 출력하세요. 다른 텍스트는 출력하지 마세요.",
        ),
        ("human", "질문: {question}\n답변: {answer}"),
    ]
)
judge_chain = judge_prompt | llm | StrOutputParser()

# judge LLM 호출은 평가용이므로 별도 트레이스로 기록하지 않음 (callbacks 미전달)
score_str = judge_chain.invoke({"question": question2, "answer": answer2})
try:
    score_value = max(0.0, min(1.0, float(score_str.strip())))
except ValueError:
    score_value = 0.5

langfuse.create_score(
    trace_id=trace_id2,
    name="llm-judge-quality",
    value=score_value,
    comment=f"LLM 자동 채점: {score_str.strip()}",
)
print(f"LLM 채점 점수: {score_value}")
print()


langfuse.flush()

<details><summary>무슨 일이 생기나</summary>

답변(200자 안팎)은 그대로인데 LLM 채점 점수가 0.95에서 **0.0**으로 떨어집니다(2026-10-21 사외 3번 모두 같음). Langfuse의 `llm-judge-quality` 점수도 0.0으로 남습니다. LLM-as-a-Judge의 점수는 답이 아니라 **Judge에게 준 채점 기준 글**이 정한다는 뜻입니다. 그래서 Judge 지시는 평가하려는 것(정확성, 관련성 등)을 분명히 적어야 합니다. 확인했으면 원래 값으로 되돌립니다.

</details>

**볼 지점**
- Judge에게 준 시스템 메시지가 곧 평가 기준입니다. 지금은 "품질"이라는 말뿐이라, 무엇을 기준으로 채점했는지 알기 어렵습니다.
- 코멘트에는 Judge가 돌려준 글이 그대로 남습니다. 점수가 이상하면 코멘트부터 봅니다.

<details><summary>강사용 메모</summary>

- 사외 실측: 원본 그대로 `LLM 채점 점수: 1.0`. Judge 응답을 `float()`로 바꾸지 못하면 **조용히 0.5**가 기록됩니다(원본 코드). 고치지 않고 두었습니다. 수강생이 물으면 짚어 줍니다.

</details>

## 7. 사용자 피드백 기록하기

실제 서비스에서는 사용자가 누른 좋아요·싫어요를 점수로 남깁니다. 이 장은 사용자가 좋아요를 눌렀다고 가정하고 `user-feedback` 점수를 1.0으로 남깁니다. 버튼이 없으므로 값을 코드에 직접 적었습니다.

원본 그대로 한 번, 값을 바꿔 한 번 실행해 두 점수를 비교합니다.

준비 코드입니다. 앞 장과 같은 모델과 답변 체인 `run_chain`을 다시 준비합니다. 실행만 합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client
from langfuse.langchain import CallbackHandler

from config.llm import get_llm

load_dotenv()

langfuse = get_client()
llm = get_llm()


def run_chain(question: str, handler: CallbackHandler) -> str:
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "한국어로 간결하게 답변하세요. 반드시 예시를 포함하세요."),
            ("human", "{question}"),
        ]
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"question": question}, config={"callbacks": [handler]})

### 7.1 사용자 피드백 점수 남기기

**✍️ CHANGE_HERE: user_feedback을 0.0(싫어요)으로 바꾸면 출력과 Langfuse의 점수가 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `1.0` 자리에 복사해 붙여 넣기

```text
0.0
```

이 셀을 다시 실행하고, Langfuse에서 두 trace의 `user-feedback` 점수를 비교합니다. 확인했으면 `1.0`으로 되돌립니다.

In [ ]:
# ── 3. 수동 평가 시뮬레이션 ──
# 실제 서비스에서는 사용자 피드백(좋아요/싫어요)을 받아 score()로 기록
print("=== 수동 평가 시뮬레이션 ===")
handler3 = CallbackHandler()
answer3 = run_chain("강화학습을 쉽게 설명해줘", handler3)
trace_id3 = handler3.last_trace_id

# 사용자가 👍를 눌렀다고 가정
# ✍️ CHANGE_HERE: user_feedback을 0.0(싫어요)으로 바꾸면 출력과 Langfuse의 점수가 어떻게 될까요?
user_feedback = 1.0
langfuse.create_score(
    trace_id=trace_id3,
    name="user-feedback",
    value=user_feedback,
    comment="사용자 좋아요",
)
print(f"답변: {answer3}")
print(f"사용자 평가: 👍 (score={user_feedback})")
print()

langfuse.flush()
print("✓ 대시보드 [Scores] 탭에서 평가 결과를 확인하세요.")

<details><summary>무슨 일이 생기나</summary>

출력이 `사용자 평가: 👍 (score=0.0)`로 바뀝니다. 👍와 코드의 코멘트 `사용자 좋아요`는 고정된 글자라 그대로이고, 바뀐 것은 `score=` 뒤 숫자뿐입니다. Langfuse에 남은 이 trace의 `user-feedback` 점수는 0.0입니다(조회 API로 확인).

점수는 숫자 값으로 남습니다. Langfuse 화면에서도 두 trace의 `user-feedback`을 값(1.0과 0.0)으로 비교합니다.

</details>

**볼 지점**
- 규칙, LLM Judge, 사용자 피드백 세 가지 점수가 모두 같은 방법(`create_score`)으로 trace에 붙었습니다. 점수를 누가 매기느냐만 다릅니다.

<details><summary>강사용 메모</summary>

- 원본 출력 문구는 "👍" 고정이라 0.0으로 바꿔도 화면 글자는 그대로입니다. 볼 지점에 적었습니다.

</details>

## 8. RAG 답변을 세 기준으로 채점하기

앞 장들을 합칩니다. `@observe`로 검색(`retrieve`)과 답변 생성(`generate`)을 기록하고, LLM Judge가 답변을 **정확성·관련성·완성도** 세 기준으로 따로 채점해 trace에 점수 세 개를 남깁니다. Judge는 정해진 형식(`EvalScore`)으로만 답하게 해서, 앞 장처럼 숫자가 아닌 글을 돌려주는 일을 막습니다.

검색은 흉내만 냅니다. 문서 세 개(LangGraph 설명, RAG 설명, 날씨 글) 중 `doc_key`로 하나를 고릅니다. 세 케이스는 이렇습니다.

- CASE A: 질문에 맞는 문서
- CASE B: 문서 없음(검색 실패)
- CASE C: 엉뚱한 문서(날씨 글)

이 장의 trace 이름은 4장과 같은 `rag-pipeline`입니다. 입력(Input) 칸이 비어 있으니 실행 시각과 점수 세 개가 붙어 있는지로 고릅니다.

준비 코드입니다. `observe`, 정해진 형식을 만드는 `BaseModel`, Langfuse 클라이언트와 모델을 준비합니다. 실행만 합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client, observe
from langfuse.langchain import CallbackHandler
from pydantic import BaseModel, Field

from config.llm import get_llm

load_dotenv()

langfuse = get_client()
llm = get_llm()

<details><summary>강사용 메모</summary>

- 원본 05 raw 사용(파일 이름에 Extra). 커리큘럼 "다차원 점수화(정확성/관련성/완성도)"를 다루는 유일한 코드라 모두 함께 하는 마지막 예제로 두었습니다(목차 5절 질문 2 제안대로). **시간이 모자라면 강사 화면 시연으로 바꿉니다.**
- `with_structured_output`(EvalScore) 사용. 사내 게이트웨이 동작은 미확인입니다. 실패하면 강사 시연으로 대체합니다.
- 사외 실측 3회: A는 세 점수 모두 1.0. B는 정확성 1.0·관련성 0.2~0.5·완성도 0.0. C는 2회가 관련성 0.5~0.7·완성도 0.0이었고, 1회는 모델이 문서를 무시하고 아는 내용으로 답해 세 점수 모두 1.0이었습니다. 원본 docstring의 "C → 낮은 정확성"과 달리 정확성은 B·C도 1.0이 나왔습니다("문서에 정보가 없다"는 답은 사실이라서). 그래서 성공 기준을 "완성도" 중심으로 썼습니다.
- **노트북에서만 고친 곳**: 원본 그대로 Jupyter 커널에서 실행하면 "실행하기" 셀이 CASE A에서 멈춥니다(2026-09-28 재현, 300초 초과). `@observe`가 함수 인자로 받은 `CallbackHandler` 객체를 입력으로 직렬화하다 끝나지 않습니다(langfuse 4.15.4 serializer가 공유 객체를 반복 탐색). `.py` 실행에서는 약 10초로 정상입니다. 그래서 `generate`·`rag-pipeline`에 `capture_input=False`를 patch했습니다. 대신 이 두 단계의 입력 칸은 비어 보입니다(출력·자식 체인의 입력은 보임).
- CHANGE_HERE는 목차대로 CASE C의 `doc_key`를 `"rag"`로 바꿉니다. 바꾸면 C의 완성도가 올라가는 쪽이 많을 것으로 봅니다(바꾼 뒤 실측은 제작 검증에서 1회).

</details>

### 8.1 문서 세 개

In [ ]:
# ── 가상 문서 저장소 ────────────────────────────────────────────────────────────
DOCS = {
    "langgraph": (
        "LangGraph는 LangChain 위에서 동작하는 상태 기반 멀티에이전트 프레임워크다. "
        "StateGraph로 노드(함수)와 엣지(전환 조건)를 정의하며, MemorySaver로 체크포인트를 저장한다. "
        "LangChain 0.x의 AgentExecutor를 대체하며 복잡한 루프·분기 흐름을 지원한다."
    ),
    "rag": (
        "RAG(Retrieval-Augmented Generation)는 외부 문서를 검색해 LLM의 답변 근거로 활용하는 기법이다. "
        "벡터 임베딩으로 유사 문서를 찾고, 해당 내용을 프롬프트 컨텍스트에 삽입한다. "
        "환각(hallucination)을 줄이고 최신 정보를 반영할 수 있다는 장점이 있다."
    ),
    "noise": (
        "오늘 서울 날씨는 맑고 기온은 23도이다. "
        "내일은 구름이 많고 오후에 비가 올 수 있다. "
        "주말에는 기온이 낮아질 전망이다."
    ),
}

### 8.2 채점 결과 형식

In [ ]:
# ── 평가 결과를 구조화하기 위한 Pydantic 모델 ──────────────────────────────────
class EvalScore(BaseModel):
    correctness: float = Field(description="사실 정확성 0.0~1.0")
    relevance: float = Field(description="질문과의 관련성 0.0~1.0")
    completeness: float = Field(description="답변 완성도 0.0~1.0")
    reason: str = Field(description="채점 근거 한 줄")

### 8.3 검색과 답변 생성 기록하기

In [ ]:
# ── RAG 단계별 함수 ─────────────────────────────────────────────────────────────


@observe(name="retrieve")
def retrieve(query: str, doc_key: str) -> str:
    """doc_key로 문서를 가져온다. 실제 환경에서는 벡터 검색으로 대체."""
    return DOCS.get(doc_key, "")


# 인자로 받은 기록 도구(handler)까지 입력으로 기록하려다 셀이 멈추므로, 입력 기록을 끕니다
@observe(name="generate", capture_input=False)
def generate(question: str, context: str, handler: CallbackHandler) -> str:
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "아래 컨텍스트만 참고해 질문에 답하세요. 컨텍스트가 없으면 모른다고 하세요.\n\n{context}",
            ),
            ("human", "{question}"),
        ]
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke(
        {"context": context, "question": question},
        config={"callbacks": [handler]},
    )


# 여기도 handler를 인자로 받으므로 입력 기록을 끕니다
@observe(name="rag-pipeline", capture_input=False)
def rag_pipeline(
    question: str, doc_key: str, handler: CallbackHandler
) -> tuple[str, str]:
    context = retrieve(question, doc_key)
    answer = generate(question, context, handler)
    return answer, context

### 8.4 세 기준으로 채점하는 Judge

In [ ]:
# ── LLM Judge ──────────────────────────────────────────────────────────────────


def judge(question: str, context: str, answer: str) -> EvalScore:
    # structured output: LLM이 EvalScore 형식으로만 응답하도록 강제
    judge_llm = llm.with_structured_output(EvalScore)
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "질문, 참고 컨텍스트, 답변을 보고 3개 항목을 0.0~1.0으로 채점하세요.\n"
                "- correctness: 컨텍스트 기준으로 답변이 사실적으로 맞는지\n"
                "- relevance: 답변이 질문에 직접 답하는지\n"
                "- completeness: 컨텍스트의 핵심 정보를 빠짐없이 담았는지",
            ),
            ("human", "질문: {question}\n\n컨텍스트: {context}\n\n답변: {answer}"),
        ]
    )
    chain = prompt | judge_llm
    return chain.invoke({"question": question, "context": context, "answer": answer})

### 8.5 테스트 케이스

**✍️ CHANGE_HERE: CASE C의 문서를 "noise"에서 "rag"로 바꾸면 C의 점수가 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `noise` 자리에 복사해 붙여 넣기

RAG 설명 문서
```text
rag
```

이 셀과 8.6 실행하기 셀을 다시 실행하고 CASE C의 점수를 바꾸기 전과 비교합니다. 확인했으면 `noise`로 되돌립니다.

In [ ]:
# ── 테스트 케이스 ───────────────────────────────────────────────────────────────

test_cases = [
    {
        "name": "CASE A — 정확한 컨텍스트",
        "question": "LangGraph가 뭔가요?",
        "doc_key": "langgraph",  # 정확한 문서
    },
    {
        "name": "CASE B — 빈 컨텍스트 (검색 실패)",
        "question": "RAG의 장점은 무엇인가요?",
        "doc_key": "missing",  # 존재하지 않는 키 → 빈 컨텍스트
    },
    {
        "name": "CASE C — 오염된 컨텍스트 (엉뚱한 문서)",
        "question": "RAG가 환각을 줄이는 원리를 설명해줘",
        # ✍️ CHANGE_HERE: CASE C의 문서를 "noise"에서 "rag"로 바꾸면 C의 점수가 어떻게 달라질까요?
        "doc_key": "noise",  # 날씨 문서 → 완전히 엉뚱함
    },
]

<details><summary>무슨 일이 생기나</summary>

CASE C의 답이 "문서에 내용이 없어 답할 수 없다"(또는 `모릅니다.`)에서 `RAG는 외부 문서를 검색해 실제 정보를 근거로 답을 만들어 환각을 줄인다…`처럼 원리를 설명하는 답으로 바뀝니다. 바꾼 뒤 점수는 정확성·관련성·완성도 모두 0.9~1.0이었습니다(2026-10-21 사외 3번).

바꾸기 전 점수는 실행마다 달랐습니다. 모른다고 답한 같은 모양의 답에 Judge가 0.0을 준 적도, 1.0을 준 적도 있습니다(문서에 충실하게 모른다고 했으니 정확하다고 본 경우). 그래서 점수만 보지 말고 `근거:` 줄을 함께 읽습니다. 확실히 달라지는 것은 **답의 내용**입니다. 같은 질문, 같은 모델이라도 검색된 문서가 맞아야 쓸모 있는 답이 나옵니다. RAG 답변이 나쁠 때는 모델보다 검색 단계부터 봅니다(trace의 `retrieve` 단계). 확인했으면 `noise`로 되돌립니다.

</details>

### 8.6 실행하기

In [ ]:
print("=== RAG 파이프라인 다차원 평가 ===\n")

for case in test_cases:
    print(f"[{case['name']}]")
    print(f"Q: {case['question']}")

    handler = CallbackHandler()
    answer, context = rag_pipeline(case["question"], case["doc_key"], handler)
    trace_id = handler.last_trace_id

    print(f"A: {answer[:120]}{'...' if len(answer) > 120 else ''}")

    # 3개 차원 채점
    score = judge(case["question"], context, answer)
    print(
        f"채점 → 정확성: {score.correctness:.1f} | 관련성: {score.relevance:.1f} | 완성도: {score.completeness:.1f}"
    )
    print(f"근거: {score.reason}")

    # Langfuse에 차원별 점수 기록
    for name, value in [
        ("correctness", score.correctness),
        ("relevance", score.relevance),
        ("completeness", score.completeness),
    ]:
        langfuse.create_score(trace_id=trace_id, name=name, value=value)

    print()

langfuse.flush()
print("✓ 대시보드 [Scores] 탭에서 케이스별 차이를 확인하세요.")

**볼 지점**
- 점수를 세 기준으로 나누면 "어디가 부족한지"가 보입니다. CASE B에서 모델이 "문서에 정보가 없어 답할 수 없다"고 답하면 정확성은 높고 완성도는 낮게 나옵니다. 모델이 문서 없이 아는 내용으로 답하면 세 점수가 모두 높게 나오기도 합니다. `근거:` 줄을 봅니다.
- CASE C는 실행마다 결과가 다를 수 있습니다. 모델이 엉뚱한 문서를 무시하고 아는 내용으로 답하면 점수가 높게 나옵니다. `근거:` 줄로 Judge가 왜 그렇게 채점했는지 봅니다.
- Tracing 목록에서 이 장의 `rag-pipeline` 세 줄을 차례로 열어 `completeness`를 비교합니다. 트리의 `retrieve`를 누르면 입력에 질문이 보여 어느 케이스인지 알 수 있습니다.

## 9. 정리

실행 기록(Trace)을 남기는 방법 세 가지와 점수(Score)를 남기는 방법 네 가지를 보았습니다.

| 장 | 방법 | Langfuse 화면에서 확인한 것 |
|---|---|---|
| 2. 체인 실행 기록하기 | `invoke`에 `callbacks`로 핸들러 넘기기 | `RunnableSequence` trace 세 개, 프롬프트 → 모델 → 파서 단계 |
| 3. 그래프 실행 기록하기 | `with_config`로 그래프에 핸들러 붙이기 | `LangGraph` trace, agent → tools → agent 노드 순서 |
| 4. 체인 밖 함수 기록하기 | `@observe(name=…)` | `rag-pipeline` 아래 검색·정리·체인 트리 |
| 5. 규칙으로 채점하기 | 코드로 판단(`has-example`) | 빠르고 재현성이 높지만 규칙이 느슨할 수 있음 |
| 6. LLM이 채점하기 | LLM Judge(`llm-judge-quality`) | 평가 기준이 흐리면 점수 근거도 흐림 |
| 7. 사용자 피드백 기록하기 | 사람의 좋아요·싫어요(`user-feedback`) | Scores 화면의 세 점수 |
| 8. RAG 답변을 세 기준으로 채점하기 | 정확성·관련성·완성도 | 어느 기준이 부족한지 나눠 보기 |

내 trace는 이름, 시각, 입력으로 찾았습니다. 기록은 몇 초 늦게 도착하므로 `flush()` 뒤 화면을 새로고침합니다.